In [2]:
import org.apache.commons.csv.*;
import java.io.*;
import java.nio.file.*;
import java.util.*;
import tech.tablesaw.api.*;
import tech.tablesaw.io.csv.*;
import tech.tablesaw.api.ColumnType;

String path = "C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\walmart-products.csv";

System.out.println("🔧 TRYING LENIENT APACHE COMMONS CSV PARSING");
System.out.println("============================================");

try (Reader reader = Files.newBufferedReader(Paths.get(path))) {

    // Use more lenient CSV format to handle malformed quotes
    CSVParser parser = new CSVParser(reader,
            CSVFormat.RFC4180
                    .withFirstRecordAsHeader()
                    .withQuote('"')
                    .withIgnoreEmptyLines(true)
                    .withAllowMissingColumnNames(true)
                    .withTrim(true));

    List<String> validLines = new ArrayList<>();
    List<CSVRecord> allRecords = parser.getRecords();
    
    System.out.println("📊 Total records found: " + allRecords.size());
    System.out.println("📋 Headers: " + parser.getHeaderNames().size() + " columns");

    // Add header manually
    validLines.add(String.join(",", parser.getHeaderNames()));

    int validCount = 0;
    int skippedCount = 0;

    for (CSVRecord record : allRecords) {
        if (record.size() == parser.getHeaderNames().size()) {
            // Properly escape any quotes in the data when reconstructing CSV
            List<String> escapedFields = new ArrayList<>();
            for (String field : record) {
                if (field == null) {
                    escapedFields.add("");
                } else if (field.contains(",") || field.contains("\"") || field.contains("\n")) {
                    // Re-escape quotes and wrap in quotes
                    String escaped = field.replace("\"", "\"\"");
                    escapedFields.add("\"" + escaped + "\"");
                } else {
                    escapedFields.add(field);
                }
            }
            validLines.add(String.join(",", escapedFields));
            validCount++;
        } else {
            System.out.println("⚠ Skipping malformed row " + record.getRecordNumber() + " with " + record.size() + " columns (expected " + parser.getHeaderNames().size() + ")");
            skippedCount++;
        }
    }

    System.out.println("✅ Valid rows: " + validCount);
    System.out.println("⚠ Skipped rows: " + skippedCount);

    String cleanedCsv = String.join("\n", validLines);

    CsvReadOptions options = CsvReadOptions.builderFromString(cleanedCsv)
            .header(true)
            .columnTypes(name -> ColumnType.STRING)
            .build();

    Table walmartTable = Table.read().usingOptions(options);

    System.out.println("\n🎉 SUCCESS! Loaded CSV with Apache Commons + Tablesaw!");
    System.out.println("Table structure:");
    System.out.println(walmartTable.structure());

} catch (Exception e) {
    System.err.println("❌ Apache Commons CSV failed: " + e.getMessage());
    System.err.println("📝 This confirms the data has complex quote issues that need custom parsing.");
    e.printStackTrace();
}

❌ Apache Commons CSV failed: org.apache.commons.csv.CSVException: Invalid character between encapsulated token and delimiter at line: 117, position: 941,291
📝 This confirms the data has complex quote issues that need custom parsing.
java.io.UncheckedIOException: org.apache.commons.csv.CSVException: Invalid character between encapsulated token and delimiter at line: 117, position: 941,291
	at org.apache.commons.io.function.Uncheck.wrap(Uncheck.java:371)
	at org.apache.commons.io.function.Uncheck.get(Uncheck.java:215)
	at org.apache.commons.csv.CSVParser$CSVRecordIterator.getNextRecord(CSVParser.java:234)
	at org.apache.commons.csv.CSVParser$CSVRecordIterator.hasNext(CSVParser.java:245)
	at java.base/java.util.Iterator.forEachRemaining(Iterator.java:132)
	at java.base/java.util.Spliterators$IteratorSpliterator.forEachRemaining(Spliterators.java:1939)
	at java.base/java.util.stream.AbstractPipeline.copyInto(AbstractPipeline.java:570)
	at java.base/java.util.stream.AbstractPipeline.wrapAnd

## When Standard Libraries Fail: Custom Parser Fallback

Even robust libraries like Apache Commons CSV can't handle severely malformed quote sequences. When libraries fail, we fall back to our custom character-by-character parser:

In [5]:
// Custom fallback parser for when standard libraries fail on malformed quotes
import java.io.*;
import java.nio.file.*;
import java.util.*;
import tech.tablesaw.api.*;
import tech.tablesaw.io.csv.*;

String walmartPath = "C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\walmart-products.csv";

System.out.println("🔧 FALLBACK: CUSTOM CSV PARSER");
System.out.println("==============================");
System.out.println("When Apache Commons CSV fails, we use character-by-character parsing...\n");

try {
    List<String> rawLines = Files.readAllLines(Paths.get(walmartPath));
    List<String> cleanedLines = new ArrayList<>();
    
    // Keep header as-is
    cleanedLines.add(rawLines.get(0));
    System.out.println("📋 Header: " + rawLines.get(0).split(",").length + " columns");
    
    int processedRows = 0;
    int validRows = 0;
    int skippedRows = 0;
    
    // Process each data row with custom parser
    for (int i = 1; i < rawLines.size(); i++) {
        try {
            String line = rawLines.get(i);
            
            // Character-by-character parsing respecting quote state
            List<String> columns = new ArrayList<>();
            boolean inQuotes = false;
            StringBuilder current = new StringBuilder();
            
            for (int j = 0; j < line.length(); j++) {
                char ch = line.charAt(j);
                
                if (ch == '"') {
                    // Handle escaped quotes: "" → "
                    if (j + 1 < line.length() && line.charAt(j + 1) == '"') {
                        current.append("\"\"");  // Preserve escaped quote
                        j++; // Skip next quote
                    } else {
                        inQuotes = !inQuotes;    // Toggle quote state
                        current.append(ch);      // Add quote to field
                    }
                } else if (ch == ',' && !inQuotes) {
                    // Comma outside quotes = field separator
                    columns.add(current.toString());
                    current.setLength(0);
                } else {
                    // Regular character (including commas inside quotes)
                    current.append(ch);
                }
            }
            columns.add(current.toString()); // Add final field
            
            // Only accept rows with expected column count
            if (columns.size() == 44) {
                // Basic cleaning: remove outer quotes, clean JSON fields
                for (int colIdx = 0; colIdx < columns.size(); colIdx++) {
                    String col = columns.get(colIdx);
                    
                    // Remove wrapper quotes added by CSV format
                    if (col.startsWith("\"") && col.endsWith("\"")) {
                        col = col.substring(1, col.length() - 1);
                    }
                    
                    // Clean known JSON columns (specs, images, reviews)
                    int[] jsonCols = {6, 7, 8, 9, 10, 14, 20, 27, 37, 38, 43};
                    final int currentColIdx = colIdx; // Make effectively final for lambda
                    boolean isJSON = Arrays.stream(jsonCols).anyMatch(jsonCol -> jsonCol == currentColIdx);
                    
                    if (isJSON && col != null) {
                        col = col
                            .replaceAll("\"\"", "\"")           // Un-escape quotes 
                            .replaceAll("\\r\\n|\\r|\\n", " ") // Remove newlines
                            .replaceAll("\\s+", " ")           // Normalize spaces
                            .trim();
                    }
                    
                    // Re-quote fields containing commas, quotes, or newlines for CSV safety
                    if (col != null && (col.contains(",") || col.contains("\"") || col.contains("\n"))) {
                        col = "\"" + col.replace("\"", "\"\"") + "\"";
                    }
                    
                    columns.set(colIdx, col != null ? col : "");
                }
                
                // Reconstruct clean CSV line with properly quoted fields
                cleanedLines.add(String.join(",", columns));
                validRows++;
            } else {
                skippedRows++;
                if (processedRows < 5) { // Show first few errors
                    System.out.println("⚠ Row " + (i+1) + ": " + columns.size() + " columns (expected 44)");
                }
            }
            
            processedRows++;
            
            if (processedRows % 250 == 0) {
                System.out.println("Processed " + processedRows + " rows... (valid: " + validRows + ", skipped: " + skippedRows + ")");
            }
            
        } catch (Exception e) {
            skippedRows++;
        }
    }
    
    System.out.println("\n📊 PARSING RESULTS:");
    System.out.println("Total processed: " + processedRows);
    System.out.println("Valid rows: " + validRows + " (" + (validRows * 100 / processedRows) + "%)");
    System.out.println("Skipped rows: " + skippedRows);
    
    // Test with Tablesaw
    String cleanedCsv = String.join("\n", cleanedLines);
    CsvReadOptions options = CsvReadOptions.builderFromString(cleanedCsv)
            .header(true)
            .columnTypes(name -> ColumnType.STRING)
            .build();
    
    Table walmartTable = Table.read().usingOptions(options);
    
    System.out.println("\n🎉 SUCCESS! Custom parser + Tablesaw worked!");
    System.out.println("Final table: " + walmartTable.rowCount() + " rows × " + walmartTable.columnCount() + " columns");
    System.out.println("\n📋 Column names (first 10):");
    walmartTable.columnNames().stream().limit(10).forEach(System.out::println);
    
} catch (Exception e) {
    System.err.println("❌ Custom parser failed: " + e.getMessage());
    e.printStackTrace();
}